# Demo 14 — Il grafico dipende anche da come lo disegniamo

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
y_colori, sr_colori = al.load_audio(AUDIO / "C-major-scale.wav", sr=SR)
A_colori = abs(al.analysis.stft(y_colori, n_fft=2048, hop_length=256))
ref_colori = A_colori.max()
db_colori = al.analysis.amplitude_to_db(A_colori, ref=ref_colori, top_db=None)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Asse lineare", "Asse logaritmico"))
for j, asse in enumerate(["linear", "log"], start=1):
    fig.add_trace(go.Heatmap(x=np.arange(db_colori.shape[1]) * 256 / sr_colori,
                             y=np.arange(db_colori.shape[0]) * sr_colori / 2048, z=db_colori,
                             colorscale="Magma", zmin=-80, zmax=0,
                             showscale=False), row=1, col=j)
    if asse == "log":
        fig.update_yaxes(type="log", range=[np.log10(80), np.log10(8000)], row=1, col=j)
    else:
        fig.update_yaxes(range=[80, 8000], row=1, col=j)
fig.update_layout(height=380, title="Il grafico dipende anche da come lo disegniamo")
fig.show()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Intervallo 40 dB", "Intervallo 80 dB"))
for j, limite in enumerate([-40, -80], start=1):
    fig.add_trace(go.Heatmap(x=np.arange(db_colori.shape[1]) * 256 / sr_colori,
                             y=np.arange(db_colori.shape[0]) * sr_colori / 2048, z=db_colori,
                             colorscale="Magma", zmin=limite, zmax=0,
                             showscale=False), row=1, col=j)
fig.update_yaxes(range=[80, 8000])
fig.update_layout(height=380, title="Quaranta o ottanta dB di intervallo: cambia la lettura")
fig.show()

# Confronto numerico prima del pavimento logaritmico.
A_debole = 0.1 * A_colori
maschera = A_colori > 1e-3 * ref_colori
np.testing.assert_allclose(20*np.log10(A_debole[maschera]/A_colori[maschera]), -20)
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=("originale: riferimento comune", "attenuato di 20 dB: riferimento comune",
                                    "originale: massimo proprio", "attenuato di 20 dB: massimo proprio"))
for riga, riferimento_comune in enumerate([True, False]):
    for colonna, (A, nome) in enumerate([(A_colori, "originale"), (A_debole, "attenuato di 20 dB")]):
        ref = ref_colori if riferimento_comune else A.max()
        db = al.analysis.amplitude_to_db(A, ref=ref, top_db=None)
        fig.add_trace(go.Heatmap(x=np.arange(db.shape[1]) * 256 / sr_colori,
                                 y=np.arange(db.shape[0]) * sr_colori / 2048, z=db,
                                 colorscale="Magma", zmin=-80, zmax=0,
                                 showscale=(riga == 1 and colonna == 1),
                                 colorbar=dict(title="dB")), row=riga + 1, col=colonna + 1)
fig.update_yaxes(range=[80, 8000])
fig.update_layout(height=640, title="Riferimento comune o massimo proprio: due letture diverse")
fig.show()
